# Lec 29-30: Tensor Programming with Einsum and Einops

Jianwen Zhu <jzhu@eecg.toronto.edu>
v2.0, revised 2026


### Overview

- We have seen Iverson's Notation
- And its implementation in Numpy
- Take Away:
  - Every value is an array
  - Operator as function of functions
  - Map-Reduce pattern: Matrix algorithms / Graph algorithms
- Generalize to N-Dimensions: Tensor functions
  - Growing bags of API and functions
    - Lots of mental burdens
    - Not portable among different implementation frameworks
    - Error prone
  - Use index notation to express products and reductions
  - Extend the same rules from matrices to higher-dimensional arrays


### Basic Einsum Functions


In [ ]:
# Companion files: run once before the lecture examples.
# Existing local files are preserved; only missing files are downloaded.
def _prepare_lecture_files():
    from pathlib import Path
    from urllib.request import urlopen
    import hashlib
    files = [('resources/test_images.npy', 'https://raw.githubusercontent.com/jianwen-zhu/2026-fall-ece326-jzhu/main/notes/resources/test_images.npy', '34e7ae78d9726db94f493500c490c20a52f08857b3b7997ce91b57d6a11c61ee')]
    root = Path.cwd().resolve()
    for name, url, expected in files:
        target = root / name
        if target.is_file():
            continue
        if not target.resolve().is_relative_to(root):
            raise RuntimeError(f"Companion path escapes the working directory: {name}")
        try:
            with urlopen(url, timeout=30) as response:
                content = response.read()
        except OSError as error:
            raise RuntimeError(f"Could not fetch {name}. Check your connection and that this lecture's files have been published.") from error
        if hashlib.sha256(content).hexdigest() != expected:
            raise RuntimeError(f"Published file {name} does not match this notebook. Reopen the latest released notebook or use its matching companion files.")
        target.parent.mkdir(parents=True, exist_ok=True)
        with target.open('xb') as output:
            output.write(content)
    print("Companion files ready.")
_prepare_lecture_files()
del _prepare_lecture_files


In [ ]:
import numpy as np

A = np.arange(9).reshape( 3, 3 )
B = np.arange(9).reshape( 3, 3 )
print( A )
print( B )


Let's look at matrix multiplication again. Here is its familiar loop-based form.


In [ ]:
N = A.shape[0]
C = np.zeros( (N,N), dtype=int )
for i in range(N) :
    for j in range(N) :
        for k in range( N ) :
            C[i,j] += A[i,k] * B[k,j]


The computation multiplies entries of A and B, then sums over k for each
output pair i,j. Einstein-style notation makes these loops implicit: the
index pattern describes the products and sums, and NumPy performs them.

So we cut the verbosity and it becomes as terse as the following: 

ik, kj -> ij

NumPy provides this notation through its `einsum` function:


```text
result = np.einsum( format, arg1, arg2, ..., argn )
```


The format string specifies the input axes and the output of the operation.
Each argument is an array; the result can be another array or a scalar.

Of course the format string has to match the dimensions:


```text
s = np.einsum( "a->", v )
T = np.einsum( "ij->ji", M )
C = np.einsum( "mn,np->mp", A, B )

assert v.ndim == len( "a" )
assert s.ndim == len( "" )

assert M.ndim == len( "ij" )
assert T.ndim == len( "ji" )

assert A.ndim == len( "mn" )
assert B.ndim == len( "np" )
assert C.ndim == len( "mp" )
```


Comparing to its nested loop counterpart, the einsum function is super compact!

Let's first see a few examples to get a flavour.


#### 2-D Matrix Diagonal Extraction


In [ ]:
d = np.zeros( (N), dtype=int )
for i in range( N ) :
    d[i] += A[i,i]

d2 = np.einsum( "ii->i", A )
print( A )
print( d )
print( d2 )


#### 2-D Matrix Trace

Recall the term "trace" in Linear Algebra?


In [ ]:
d = 0
for i in range( N ) :
    d += A[i,i]
d2 = np.einsum( "ii->", A )

print( A )
print( d )
print( d2 )
assert np.allclose( d, d2 )


#### Three Argument Example


In [ ]:
v = np.arange( N )
d = 0
for s in range( N ) :
    for t in range( N ) :
        d += v[s] * A[s,t] * v[t]
d2 = np.einsum( "s,st,t->", v, A, v )

print( v )
print( d )
print( d2 )
assert np.allclose( d, d2 )


#### Batched Outer Product


In [ ]:
d = np.zeros( (N,N,N), dtype=int )
for b in range( N ) :
    for i in range( N ) :
        for j in range( N ) :
            d[b,i,j] += A[b,i] * B[b,j]
d2 = np.einsum( "bi,bj->bij", A, B )

print( d2 )
assert np.allclose( d, d2 ) 


#### Semantics

The examples express several familiar operations through a shared index
notation. How does an einsum expression specify its result?

We distinguish two types of indices:

1. Free Indices:

   Indices *present* in the output indices;
   
1. Summation (Reduction) Indices: 

    Indices *disappeared* in the output indices.

Typically, free indices are in the outer loop, summation indices are in the inner loop.
As you recall, summation (reduction) carries out *dimension reduction*: that dimension will be GONE at the output. 

Now let's go back to the previous examples again and identify the free and summation indices. 

The order of labels in each input specifies which label belongs to each axis.
The order after the arrow specifies the output axes: `ij->ji` transposes a
matrix. Reusing a label identifies coordinates that must match; a repeated
label within one operand selects a diagonal.

The expression specifies products and reductions, while an implementation can
choose how to carry them out. For several operands, choosing the order of
intermediate products can greatly affect time and memory. Floating-point
rounding can also depend on the order of arithmetic operations.

When the arrow is omitted, NumPy takes the labels that appear exactly once
and orders them alphabetically. Thus `ij,jk` is equivalent to `ij,jk->ik`.
Writing the output labels explicitly makes the intended axis order clear.


In [ ]:
print( np.einsum( 'ji', A ) )
print( np.einsum( 'ji->ij', A ) )


#### Wrong Uses


```text
#    Bad. Number of input index groups doesn't match number of
#         arguments.
np.einsum("ab,bc->ac", A)
 
#    Bad. Indexes must be ASCII upper/lowercase letters.
np.einsum("012,1^%->:;?", A, B)
 
#    Bad. Argument 0 has 3 dimensions but only 2 indexes are
#         given.
A = np.random.normal(size = (2,3,4))
B = np.random.normal(size = (4,5,6))
np.einsum("ab,bcd->a", A, B)
 
#    Bad. One of the output indexes isn't in the set of all
#         input indexes.
np.einsum("ab,bc->acz", A, B)
 
#    Bad. Output has a repeated index.
np.einsum("ab,bc->baa", A, B)


#    Bad. Mismatches in the sizes of input argument axes
#         that are labelled with the same index.
A = np.random.normal(size = (2,3,4))
B = np.random.normal(size = (3,4,5))
np.einsum("ckj,cqq->c", A, B)
```


#### A Shared Notation for Familiar Operations

Einsum provides a systematic vocabulary for many operations. Equivalent
computations can have different expressions: index names can be changed
consistently, and a computation can be split into intermediate steps.
The benefit is a small set of rules that applies across many named functions.

Let's first look at 1D Vector functions.

| Einsum             | Numpy Equivalent  | Description         |
| ------------------ | ----------------- | ---                 |
| ('i', A)           | A                 | return a view of A  |
| ('i->', A)         | sum(A)            | sum all elements of A |
| ('i,i->i', A, B)   | A*B               | element-wise mult of two vectors |
| ('i,i->', A, B)    | inner(A, B)       | inner product of two vectors |
| ('i,j->ij', A, B)  | outer(A,B)        | outer product of two vectors |

Now let's look at 2D Matrix functions over matrices with compatible shapes.

| Einsum             | Numpy Equivalent  | Description         |
| ------------------ | ----------------- | ---                 |
| ('ij', A)          | A                 | a View of itself    |
| ('ji', A)          | A.T               | Transpose | 
| ('ii->i', A)       | diag(A)           | main diagnoal of a matrix |
| ('ij->', A)        | sum(A)            | reduction on ALL elements |
| ('ij->j', A)       | sum(A,axis=0)     | reduction on ALL rows |
| ('ij->i', A)       | sum(A,axis=1)     | reduction on ALL columns | 
| ('ij,ij->ij', A, B) |	A * B	         | Hadamard product: element-wise multiplication of A and B |
| ('ij,ji->ij', A, B) |	A * B.T	         | element-wise multiplication of A and B.T |
| ('ij,jk', A, B)	  | dot(A, B)	     | matrix multiplication of A and B |
| ('ij,kj->ik', A, B) | inner(A, B)	     | inner product of A and B |
| ('ij,kj->ikj', A, B) | A[:, None] * B	  | each row of A multiplied by B |
| ('ij,kl->ijkl', A, B)	| A[:, :, None, None] * B	| each value of A multiplied by B|


#### Marching to Higher Dimensions

Einsum really shines when dealing with tensors beyond 2 dimensions.

- Batch Matrix Multiplication


In [ ]:
# Batch matrix multiplication
X = np.arange(24).reshape(2, 3, 4)
Y = np.arange(40).reshape(2, 4, 5)

A = np.einsum('ijk, ikl->ijl', X, Y)

print(A)


In this example, i,j,l are free indices, and k is the summation index. We extend our familiar matrix multiplication like behavior to three dimensional arrays. It can be thought of performing matrix multiplications on batches of data, hence the name. But in Einstin notation, there is really not much difference.

- Tensor Contraction


In [ ]:
X = np.random.rand( 2,3,5,7 )
Y = np.random.rand( 11,13,3,17,5 )
A = np.einsum('pqrs,tuqvr->pstuv', X, Y)
print( A.shape )


### Einops and Named Axes

NumPy's einsum describes products and sums using single-letter axis labels.
Einops complements this with named-axis notation and a small set of operations:

- `einsum`: products and sums, with descriptive axis names.
- `rearrange`: reorder, combine, and split axes while preserving the elements.
- `reduce`: combine elements using a reduction such as sum, mean, min, or max.
- `repeat`: repeat elements along existing or new axes.

Alex Rogozhnikov's Einops package uses explicit output patterns. Its einsum
function separates axis names with spaces and places the pattern after the
input arrays. Compare the following examples with their NumPy counterparts.


Install the packages into your notebook's kernel if needed:


In [ ]:
%pip install numpy "einops>=0.6"


The upstream repository also contains the standalone tutorial:


```text
!git clone https://github.com/arogozhnikov/einops.git
```


In [ ]:
import numpy as np
from einops import einsum

# Return to two-dimensional matrices after the higher-dimensional examples.
A = np.arange(9).reshape(3, 3)
B = np.arange(9).reshape(3, 3)
einsum( A, 'row col -> row' )


In [ ]:
einsum( A, B, 'i k, k j -> i j' )


The visual tutorial below develops `rearrange`, `reduce`, and `repeat` using
the same character images. Follow how each pattern changes their arrangement.


### Einops visual tutorial: rearrange, reduce, repeat

The following complete tutorial is Alex Rogozhnikov's
[Einops tutorial, part 1: basics](https://einops.rocks/1-einops-basics/).
Its six character images make axis composition and decomposition visible.
The original examples and exercises are retained. The small display helper is
included below. The six images are supplied in `resources/test_images.npy`
alongside this notebook; keep the resources folder beside it when copying it.

Source: [upstream notebook, pinned revision](https://github.com/arogozhnikov/einops/blob/1f2268c55a25e1d422680a2f22a4c71f4faa7f3d/docs/1-einops-basics.ipynb).
The tutorial is reproduced under the MIT license below.


<!-- <img src='http://arogozhnikov.github.io/images/einops/einops_logo_350x350.png' height="80" /> -->


### Welcome to einops-land!

We don't write 


```python
y = x.transpose(0, 2, 3, 1)
```


We write comprehensible code


```python
y = rearrange(x, 'b c h w -> b h w c')
```


`einops` supports widely used tensor packages (such as `numpy`, `pytorch`, `jax`, `tensorflow`), and extends them.


### What's in this tutorial?

- fundamentals: reordering, composition and decomposition of axes
- operations: `rearrange`, `reduce`, `repeat`
- how much you can do with a single operation!


### Preparations


In [ ]:
# Pillow lets the display helper below render NumPy arrays as images.
%pip install pillow -q


In [ ]:
# Examples are given for numpy. This code also setups ipython/jupyter/jupyterlite
# so that numpy arrays in the output are displayed as images
import numpy
from IPython import get_ipython
from PIL.Image import fromarray

def display_np_arrays_as_images():
    def np_to_png(a):
        if a.ndim in (2, 3) and (a.ndim == 2 or a.shape[-1] in (3, 4)):
            return fromarray(numpy.array(numpy.clip(a, 0, 1) * 255, dtype="uint8"))._repr_png_()
        return None
    get_ipython().display_formatter.formatters["image/png"].for_type(numpy.ndarray, np_to_png)

display_np_arrays_as_images()


### Load a batch of images to play with


In [ ]:
ims = numpy.load("resources/test_images.npy", allow_pickle=False)
# There are 6 images of shape 96x96 with 3 color channels packed into tensor
print(ims.shape, ims.dtype)


In [ ]:
# display the first image (whole 4d tensor can't be rendered)
ims[0]


In [ ]:
# second image in a batch
ims[1]


In [ ]:
# we'll use three operations
from einops import rearrange, reduce, repeat


In [ ]:
# rearrange, as the name suggests, rearranges elements
# below we swapped height and width.
# In other words, transposed first two axes (dimensions)
rearrange(ims[0], "h w c -> w h c")


In [ ]:
# we could use more verbose names for axes, and result is the same:
rearrange(ims[0], "height width color -> width height color")
# when you operate on same set of axes many times,
# you usually come up with short names.
# That's what we do throughout tutorial - we'll use b (for batch), h, w, and c


### Composition of axes
transposition is very common and useful, but let's move to other capabilities provided by einops


In [ ]:
# einops allows seamlessly composing batch and height to a new height dimension
# We just rendered all images by collapsing to 3d tensor!
rearrange(ims, "b h w c -> (b h) w c")


In [ ]:
# or compose a new dimension of batch and width
rearrange(ims, "b h w c -> h (b w) c")


In [ ]:
# resulting dimensions are computed very simply
# length of newly composed axis is a product of components
# [6, 96, 96, 3] -> [96, (6 * 96), 3]
rearrange(ims, "b h w c -> h (b w) c").shape


In [ ]:
# we can compose more than two axes.
# let's flatten 4d array into 1d, resulting array has as many elements as the original
rearrange(ims, "b h w c -> (b h w c)").shape


### Decomposition of axis


In [ ]:
# decomposition is the inverse process - represent an axis as a combination of new axes
# several decompositions possible, so b1=2 is to decompose 6 to b1=2 and b2=3
rearrange(ims, "(b1 b2) h w c -> b1 b2 h w c ", b1=2).shape


In [ ]:
# finally, combine composition and decomposition:
rearrange(ims, "(b1 b2) h w c -> (b1 h) (b2 w) c ", b1=2)


In [ ]:
# slightly different composition: b1 is merged with width, b2 with height
# ... so letters are ordered by w then by h
rearrange(ims, "(b1 b2) h w c -> (b2 h) (b1 w) c ", b1=2)


In [ ]:
# move part of width dimension to height.
# we should call this width-to-height as image width shrunk by 2 and height doubled.
# but all pixels are the same!
# Can you write reverse operation (height-to-width)?
rearrange(ims, "b h (w w2) c -> (h w2) (b w) c", w2=2)


### Order of axes matters


In [ ]:
# compare with the next example
rearrange(ims, "b h w c -> h (b w) c")


In [ ]:
# order of axes in composition is different
# rule is just as for digits in the number: leftmost digit is the most significant,
# while neighboring numbers differ in the rightmost axis.

# you can also think of this as lexicographic sort
rearrange(ims, "b h w c -> h (w b) c")


In [ ]:
# what if b1 and b2 are reordered before composing to width?
rearrange(ims, "(b1 b2) h w c -> h (b1 b2 w) c ", b1=2)  # produces 'einops'
rearrange(ims, "(b1 b2) h w c -> h (b2 b1 w) c ", b1=2)  # produces 'eoipns'


### Meet einops.reduce

In einops-land you don't need to guess what happened


```python
x.mean(-1)
```


Because you write what the operation does


```python
reduce(x, 'b h w c -> b h w', 'mean')
```


if axis is not present in the output — you guessed it — axis was reduced.


In [ ]:
# average over batch
reduce(ims, "b h w c -> h w c", "mean")


In [ ]:
# the previous is identical to familiar:
ims.mean(axis=0)
# but is so much more readable


In [ ]:
# Example of reducing of several axes
# besides mean, there are also min, max, sum, prod
reduce(ims, "b h w c -> h w", "min")


In [ ]:
# this is mean-pooling with 2x2 kernel
# image is split into 2x2 patches, each patch is averaged
reduce(ims, "b (h h2) (w w2) c -> h (b w) c", "mean", h2=2, w2=2)


In [ ]:
# max-pooling is similar
# result is not as smooth as for mean-pooling
reduce(ims, "b (h h2) (w w2) c -> h (b w) c", "max", h2=2, w2=2)


In [ ]:
# yet another example. Can you compute result shape?
reduce(ims, "(b1 b2) h w c -> (b2 h) (b1 w)", "mean", b1=2)


### Stack and concatenate


In [ ]:
# rearrange can also take care of lists of arrays with the same shape
x = list(ims)
print(type(x), "with", len(x), "tensors of shape", x[0].shape)
# that's how we can stack inputs
# "list axis" becomes first ("b" in this case), and we left it there
rearrange(x, "b h w c -> b h w c").shape


In [ ]:
# but new axis can appear in the other place:
rearrange(x, "b h w c -> h w c b").shape


In [ ]:
# that's equivalent to numpy stacking, but written more explicitly
numpy.array_equal(rearrange(x, "b h w c -> h w c b"), numpy.stack(x, axis=3))


In [ ]:
# ... or we can concatenate along axes
rearrange(x, "b h w c -> h (b w) c").shape


In [ ]:
# which is equivalent to concatenation
numpy.array_equal(rearrange(x, "b h w c -> h (b w) c"), numpy.concatenate(x, axis=1))


### Addition or removal of axes

You can write 1 to create a new axis of length 1. Similarly you can remove such axis.

There is also a synonym `()` that you can use. That's a composition of zero axes and it also has a unit length.


In [ ]:
x = rearrange(ims, "b h w c -> b 1 h w 1 c")  # functionality of numpy.expand_dims
print(x.shape)
print(rearrange(x, "b 1 h w 1 c -> b h w c").shape)  # functionality of numpy.squeeze


In [ ]:
# compute max in each image individually, then show a difference
x = reduce(ims, "b h w c -> b () () c", "max") - ims
rearrange(x, "b h w c -> h (b w) c")


### Repeating elements

Third operation we introduce is `repeat`


In [ ]:
# repeat along a new axis. New axis can be placed anywhere
repeat(ims[0], "h w c -> h new_axis w c", new_axis=5).shape


In [ ]:
# shortcut
repeat(ims[0], "h w c -> h 5 w c").shape


In [ ]:
# repeat along w (existing axis)
repeat(ims[0], "h w c -> h (repeat w) c", repeat=3)


In [ ]:
# repeat along two existing axes
repeat(ims[0], "h w c -> (2 h) (2 w) c")


In [ ]:
# order of axes matters as usual - you can repeat each element (pixel) 3 times
# by changing order in parenthesis
repeat(ims[0], "h w c -> h (w repeat) c", repeat=3)


Note: `repeat` operation covers functionality identical to `numpy.repeat`, `numpy.tile` and actually more than that.


### Reduce ⇆ repeat

reduce and repeat are like opposite of each other: first one reduces amount of elements, second one increases.

In the following example each image is repeated first, then we reduce over new axis to get back original tensor. Notice that operation patterns are "reverse" of each other


In [ ]:
repeated = repeat(ims, "b h w c -> b h new_axis w c", new_axis=2)
reduced = reduce(repeated, "b h new_axis w c -> b h w c", "min")
assert numpy.array_equal(ims, reduced)


### Fancy examples in random order

These examples combine the operations introduced above. Predict the layout
or shape before running each cell, then use the image to check your interpretation.

(a.k.a. mad designer gallery)


In [ ]:
# interweaving pixels of different pictures
# all letters are observable
rearrange(ims, "(b1 b2) h w c -> (h b1) (w b2) c ", b1=2)


In [ ]:
# interweaving along vertical for couples of images
rearrange(ims, "(b1 b2) h w c -> (h b1) (b2 w) c", b1=2)


In [ ]:
# interweaving lines for couples of images
# exercise: achieve the same result without einops in your favourite framework
reduce(ims, "(b1 b2) h w c -> h (b2 w) c", "max", b1=2)


In [ ]:
# color can be also composed into dimension
# ... while image is downsampled
reduce(ims, "b (h 2) (w 2) c -> (c h) (b w)", "mean")


In [ ]:
# disproportionate resize
reduce(ims, "b (h 4) (w 3) c -> (h) (b w)", "mean")


In [ ]:
# spilt each image in two halves, compute mean of the two
reduce(ims, "b (h1 h2) w c -> h2 (b w)", "mean", h1=2)


In [ ]:
# split in small patches and transpose each patch
rearrange(ims, "b (h1 h2) (w1 w2) c -> (h1 w2) (b w1 h2) c", h2=8, w2=8)


In [ ]:
# stop me someone!
rearrange(ims, "b (h1 h2 h3) (w1 w2 w3) c -> (h1 w2 h3) (b w1 h2 w3) c", h2=2, w2=2, w3=2, h3=2)


In [ ]:
rearrange(ims, "(b1 b2) (h1 h2) (w1 w2) c -> (h1 b1 h2) (w1 b2 w2) c", h1=3, w1=3, b2=3)


In [ ]:
# patterns can be arbitrarily complicated
reduce(ims, "(b1 b2) (h1 h2 h3) (w1 w2 w3) c -> (h1 w1 h3) (b1 w2 h2 w3 b2) c", "mean", h2=2, w1=2, w3=2, h3=2, b2=2)


In [ ]:
# subtract background in each image individually and normalize
# pay attention to () - this is composition of 0 axis, a dummy axis with 1 element.
im2 = reduce(ims, "b h w c -> b () () c", "max") - ims
im2 /= reduce(im2, "b h w c -> b () () c", "max")
rearrange(im2, "b h w c -> h (b w) c")


In [ ]:
# pixelate: first downscale by averaging, then upscale back using the same pattern
averaged = reduce(ims, "b (h h2) (w w2) c -> b h w c", "mean", h2=6, w2=8)
repeat(averaged, "b h w c -> (h h2) (b w w2) c", h2=6, w2=8)


In [ ]:
rearrange(ims, "b h w c -> w (b h) c")


In [ ]:
# let's bring color dimension as part of horizontal axis
# split width into groups of 3 and move the within-group axis into height
reduce(ims, "b (h h2) (w w2) c -> (h w2) (b w c)", "mean", h2=3, w2=3)


### Ok, numpy is fun, but how do I use einops with some other framework?

If that's what you've done with `ims` being numpy array:


```python
rearrange(ims, 'b h w c -> w (b h) c')
```


That's how you adapt the code for other frameworks:


```python
# pytorch:
rearrange(ims, 'b h w c -> w (b h) c')
# tensorflow:
rearrange(ims, 'b h w c -> w (b h) c')
# gluon:
rearrange(ims, 'b h w c -> w (b h) c')
# cupy:
rearrange(ims, 'b h w c -> w (b h) c')
# jax:
rearrange(ims, 'b h w c -> w (b h) c')
# paddle:
rearrange(ims, 'b h w c -> w (b h) c')

...well, you got the idea.
```


Einops allows backpropagation as if all operations were native to framework.
Operations do not change when moving to another framework - einops notation is universal


## Summary

- `rearrange` doesn't change number of elements and covers different numpy functions (like `transpose`, `reshape`, `stack`, `concatenate`,  `squeeze` and `expand_dims`)
- `reduce` combines same reordering syntax with reductions (`mean`, `min`, `max`, `sum`, `prod`, and any others)
- `repeat` additionally covers repeating and tiling
- composition and decomposition of axes are a corner stone, they can and should be used together



- [Second part of tutorial](https://github.com/arogozhnikov/einops/tree/main/docs) shows how einops works with other frameworks
- [Third part of tutorial](https://arogozhnikov.github.io/einops/pytorch-examples.html) shows how to improve your DL code with einops


### Reading the axis rules

Read a pattern as *input axes → output axes*. The function tells us what
operation connects the two sides. Axis names identify coordinates, not values:
`height width` names two axes, while `height_width` names one.

For `rearrange`, `repeat`, and `reduce`:

- Each named axis occurs at most once on each side. A name appearing on both
  sides refers to the same elementary axis and keeps its length.
- Parentheses combine axes into one dimension: `(h w)` has length `h * w`.
  On the input side, they split a dimension into factors. Supply enough lengths
  to determine the split; einops rejects an ambiguous factorization.
- Order matters. In `(h w)`, `w` changes fastest; `(w h)` enumerates the same
  coordinates in a different order. Parentheses do not multiply array values.
- A literal `1` (or empty parentheses `()`) denotes a singleton dimension.
  `rearrange` can insert or remove it without changing the number of elements.

| Operation | Rule for named axes | What happens to values? |
|-----------|---------------------|-------------------------|
| `rearrange` | Same names on both sides | Reorganize existing elements |
| `repeat` | All input names retained; new output names allowed | Repeat values along each new axis |
| `reduce` | Output names drawn from input names | Aggregate along omitted axes using the specified operation |

For example, =repeat(x, 'i -> r i', r=3)= means `y[r, i] ` x[i]=.
The input does not depend on `r`, so every row has the same values. The new
axis can also be combined with an existing one: `i -> (r i)` tiles the vector,
whereas `i -> (i r)` repeats each element. Writing `i -> i i` is invalid:
repetition introduces a *new axis name*, not a second occurrence of `i`.
Every `rearrange` transformation can also be expressed with `repeat`; the
separate name makes the intention to preserve elements explicit.


#### How einsum differs

`einsum` describes indexed products and sums across one or more arrays.
With an explicit output pattern, labels omitted from the output are summed
over. Shared labels align coordinates across operands; a label repeated within
one operand selects a diagonal. Output labels must be distinct and must occur
in an input. Axis splitting and merging with parentheses belong to
`rearrange`, not `einsum`.


In [ ]:
M = np.arange(6).reshape(2, 3)
N = np.arange(12).reshape(3, 4)

# C[i, k] = sum over j of M[i, j] * N[j, k]
C = einsum(M, N, 'row shared, shared column -> row column')
assert np.array_equal(C, np.einsum('ij,jk->ik', M, N))
C


NumPy uses single-letter labels and puts the pattern first. Einops uses
space-separated names and puts the pattern last. Both examples above describe
the same indexed computation. Unlike `reduce`, `einsum` combines multiple
operands by multiplication and reduces by summation; `reduce` operates on one
array and lets us choose `sum`, `mean`, `max`, or another supported reduction.


### Einops tutorial license


```text
MIT License

Copyright (c) 2018 Alex Rogozhnikov

Permission is hereby granted, free of charge, to any person obtaining a copy
of this software and associated documentation files (the "Software"), to deal
in the Software without restriction, including without limitation the rights
to use, copy, modify, merge, publish, distribute, sublicense, and/or sell
copies of the Software, and to permit persons to whom the Software is
furnished to do so, subject to the following conditions:

The above copyright notice and this permission notice shall be included in all
copies or substantial portions of the Software.

THE SOFTWARE IS PROVIDED "AS IS", WITHOUT WARRANTY OF ANY KIND, EXPRESS OR
IMPLIED, INCLUDING BUT NOT LIMITED TO THE WARRANTIES OF MERCHANTABILITY,
FITNESS FOR A PARTICULAR PURPOSE AND NONINFRINGEMENT. IN NO EVENT SHALL THE
AUTHORS OR COPYRIGHT HOLDERS BE LIABLE FOR ANY CLAIM, DAMAGES OR OTHER
LIABILITY, WHETHER IN AN ACTION OF CONTRACT, TORT OR OTHERWISE, ARISING FROM,
OUT OF OR IN CONNECTION WITH THE SOFTWARE OR THE USE OR OTHER DEALINGS IN THE
SOFTWARE.
```


### Take Away

Einstein's notation, in the same spirit of Iverson's APL notation, encourages abstract thinking, and reduces a plethora of seemingly different things into a handful of concepts.

With engineering efforts progressed through Numpy and Einops (2018), the abstract thinking can translate to real world impact. For example, Einops not only admits extremely terse expression of your design intent, it can also connect transparently to different "backends". This not only incldues numpy, but also different deep learning frameworks, which map the program to different hardware, some can be as large as an entire cluster in a data center with tens of thought GPUs.
